# PySpark drills 7: aggregations and windows

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-07-aggregations-and-windows.ipynb)

Practice for [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb). Each exercise uses the `orders` and `events` tables, or a small input from its data cell.

Write your answer in the empty cell under each exercise, run it, and compare your output with the expected output.
The reference answer is hidden under the exercise; open it when you are done or stuck.

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC) and `events` (one row per user event; one event is an exact retry): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
from pyspark.sql import Window

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

events = spark.createDataFrame([
    (1, "view", datetime(2026, 1, 6, 9, 0)),
    (1, "add_to_cart", datetime(2026, 1, 6, 9, 2)),
    (1, "purchase", datetime(2026, 1, 6, 9, 10)),
    (2, "view", datetime(2026, 1, 7, 10, 0)),
    (2, "purchase", datetime(2026, 1, 7, 10, 40)),
    (3, "view", datetime(2026, 1, 6, 11, 15)),
    (3, "view", datetime(2026, 1, 8, 11, 20)),
    (4, "add_to_cart", datetime(2026, 1, 9, 20, 5)),
    (4, "purchase", datetime(2026, 1, 10, 8, 30)),
    (5, "view", datetime(2026, 1, 11, 14, 45)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),
    (5, "purchase", datetime(2026, 1, 12, 12, 30)),
    (5, "view", datetime(2026, 1, 12, 9, 0)),  # an SDK retry committed this event twice
], "user_id int, event string, ts timestamp")

## Exercise 1: Customer summary

**Difficulty:** Easy · about 5 min

Per customer, return `customer_id`, `orders` (count), `revenue` (sum of `amount`) and `avg_order` (average amount
rounded to 2 decimals), ordered by `customer_id`.

**Expected output**

| customer_id | orders | revenue | avg_order |
| --- | --- | --- | --- |
| 1 | 3 | 234.50 | 78.17 |
| 2 | 2 | 315.25 | 157.63 |
| 3 | 2 | 459.00 | 229.50 |
| 4 | 2 | 1269.90 | 634.95 |
| 5 | 1 | 74.00 | 74.00 |
| 6 | 2 | 306.40 | 153.20 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = orders.groupBy("customer_id").agg(
    F.count("*").alias("orders"), F.sum("amount").alias("revenue"), F.round(F.avg("amount"), 2).alias("avg_order")
).orderBy("customer_id")
result.show()

## Exercise 2: Orders per status as columns

**Difficulty:** Medium · about 10 min

Return one row per customer with the number of orders in each status as columns `placed`, `paid`, `shipped` and
`returned` (0 when there are none), ordered by `customer_id`.

**Expected output**

| customer_id | placed | paid | shipped | returned |
| --- | --- | --- | --- | --- |
| 1 | 0 | 1 | 1 | 1 |
| 2 | 1 | 1 | 0 | 0 |
| 3 | 0 | 1 | 1 | 0 |
| 4 | 0 | 0 | 1 | 1 |
| 5 | 0 | 1 | 0 | 0 |
| 6 | 1 | 0 | 1 | 0 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
statuses = ["placed", "paid", "shipped", "returned"]
result = orders.groupBy("customer_id").pivot("status", statuses).count().fillna(0, statuses).orderBy("customer_id")
result.show()

## Exercise 3: Top two orders per customer

**Difficulty:** Medium · about 10 min

For each customer, keep the two largest orders by `amount` (ties broken by the smaller `order_id`). Return
`customer_id`, `order_id`, `amount` and `rank`, ordered by `customer_id` then `rank`.

**Expected output**

| customer_id | order_id | amount | rank |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | 1 |
| 1 | 2 | 84.50 | 2 |
| 2 | 4 | 260.00 | 1 |
| 2 | 5 | 55.25 | 2 |
| 3 | 7 | 310.00 | 1 |
| 3 | 6 | 149.00 | 2 |
| 4 | 9 | 1180.00 | 1 |
| 4 | 8 | 89.90 | 2 |
| 5 | 10 | 74.00 | 1 |
| 6 | 11 | 210.00 | 1 |
| 6 | 12 | 96.40 | 2 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
by_amount = Window.partitionBy("customer_id").orderBy(F.desc("amount"), "order_id")
result = (
    orders.withColumn("rank", F.row_number().over(by_amount))
    .where("rank <= 2")
    .select("customer_id", "order_id", "amount", "rank")
    .orderBy("customer_id", "rank")
)
result.show()

## Exercise 4: Running total per customer

**Difficulty:** Medium · about 10 min

For customers 1 to 3, return each order with `customer_id`, `order_id`, `amount` and `running_total`: the sum of the
customer's amounts up to and including this order, in `order_ts` order. Order the output by `customer_id` and
`order_ts`.

**Expected output**

| customer_id | order_id | amount | running_total |
| --- | --- | --- | --- |
| 1 | 1 | 120.00 | 120.00 |
| 1 | 2 | 84.50 | 204.50 |
| 1 | 3 | 30.00 | 234.50 |
| 2 | 4 | 260.00 | 260.00 |
| 2 | 5 | 55.25 | 315.25 |
| 3 | 6 | 149.00 | 149.00 |
| 3 | 7 | 310.00 | 459.00 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
so_far = Window.partitionBy("customer_id").orderBy("order_ts", "order_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
result = (
    orders.where("customer_id IN (1, 2, 3)")
    .withColumn("running_total", F.sum("amount").over(so_far))
    .orderBy("customer_id", "order_ts")
    .select("customer_id", "order_id", "amount", "running_total")
)
result.show()

## Exercise 5: Sessions per user

**Difficulty:** Hard · about 20 min

Remove exact duplicate events, then split each user's events into sessions: a new session starts when more than 30
minutes have passed since the user's previous event. Return `user_id` and `sessions`, ordered by `user_id`.

**Expected output**

| user_id | sessions |
| --- | --- |
| 1 | 1 |
| 2 | 2 |
| 3 | 2 |
| 4 | 2 |
| 5 | 3 |

In [ ]:
# This exercise uses the sample tables created above.

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
user_time = Window.partitionBy("user_id").orderBy("ts")
flagged = (
    events.dropDuplicates()
    .withColumn("gap_minutes", (F.col("ts").cast("long") - F.lag("ts").over(user_time).cast("long")) / 60)
    .withColumn("new_session", F.when(F.col("gap_minutes").isNull() | (F.col("gap_minutes") > 30), 1).otherwise(0))
)
result = flagged.groupBy("user_id").agg(F.sum("new_session").alias("sessions")).orderBy("user_id")
result.show()

**Next:** [PySpark 8: nested and semi-structured data](pyspark-08-nested-and-semi-structured-data.ipynb)